# YOLOv8 Training — Football Players Detection

This notebook trains a YOLOv8 model on the **football-players-detection** dataset (Roboflow export) stored in your Google Drive.

**Dataset structure expected (as in your Drive):**
```
football-players-detection/
├── data.yaml
├── README.dataset.txt
├── README.roboflow.txt
├── train/
│   ├── images/
│   └── labels/
├── valid/
│   ├── images/
│   └── labels/
└── test/
    ├── images/
    └── labels/
```

**Steps:** Mount Drive → Install Ultralytics → Fix `data.yaml` paths → Train (50 epochs) → Validate → Run inference on test images → Save results back to Drive.

In [1]:
# 1. Check GPU
!nvidia-smi

Sat Jul 11 10:17:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 3. Set dataset path

Update `DATASET_DIR` below to point to the exact folder in your Drive that contains `data.yaml`, `train/`, `valid/`, `test/` (the folder shown in your screenshot).

In [3]:
import os

# TODO: update this path to match where the dataset folder lives in your Drive
DATASET_DIR = "/content/drive/MyDrive/football-players-detection"

assert os.path.exists(DATASET_DIR), f"Dataset folder not found: {DATASET_DIR}. Update DATASET_DIR above."
print("Contents of dataset folder:")
print(os.listdir(DATASET_DIR))

Contents of dataset folder:
['data.yaml', 'README.dataset.txt', 'README.roboflow.txt', 'test', 'train', 'valid']


## 4. Install Ultralytics (YOLOv8)

In [4]:
!pip install ultralytics -q
import ultralytics
ultralytics.checks()

Ultralytics 8.4.92 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 46.7/112.6 GB disk)


## 5. Inspect / fix `data.yaml`

Roboflow's `data.yaml` sometimes points to relative paths that don't resolve correctly in Colab. This cell reads the existing `data.yaml`, rewrites the `train`, `val`, and `test` paths to absolute paths pointing at your Drive folders, and saves a corrected copy (`data_colab.yaml`) that we'll use for training.

In [5]:
import yaml

yaml_path = os.path.join(DATASET_DIR, "data.yaml")

with open(yaml_path, "r") as f:
    data_cfg = yaml.safe_load(f)

print("Original data.yaml:")
print(data_cfg)

data_cfg["train"] = os.path.join(DATASET_DIR, "train", "images")
data_cfg["val"] = os.path.join(DATASET_DIR, "valid", "images")
data_cfg["test"] = os.path.join(DATASET_DIR, "test", "images")

colab_yaml_path = os.path.join(DATASET_DIR, "data_colab.yaml")
with open(colab_yaml_path, "w") as f:
    yaml.dump(data_cfg, f, default_flow_style=False)

print("\nUpdated data_colab.yaml:")
print(data_cfg)
print(f"\nNumber of classes: {data_cfg.get('nc')}")
print(f"Class names: {data_cfg.get('names')}")

Original data.yaml:
{'train': '../train/images', 'val': '../valid/images', 'test': '../test/images', 'nc': 4, 'names': ['ball', 'goalkeeper', 'player', 'referee'], 'roboflow': {'workspace': 'roboflow-jvuqo', 'project': 'football-players-detection-3zvbc', 'version': 20, 'license': 'CC BY 4.0', 'url': 'https://universe.roboflow.com/roboflow-jvuqo/football-players-detection-3zvbc/dataset/20'}}

Updated data_colab.yaml:
{'train': '/content/drive/MyDrive/football-players-detection/train/images', 'val': '/content/drive/MyDrive/football-players-detection/valid/images', 'test': '/content/drive/MyDrive/football-players-detection/test/images', 'nc': 4, 'names': ['ball', 'goalkeeper', 'player', 'referee'], 'roboflow': {'workspace': 'roboflow-jvuqo', 'project': 'football-players-detection-3zvbc', 'version': 20, 'license': 'CC BY 4.0', 'url': 'https://universe.roboflow.com/roboflow-jvuqo/football-players-detection-3zvbc/dataset/20'}}

Number of classes: 4
Class names: ['ball', 'goalkeeper', 'player

## 6. Sanity check — count images/labels in each split

In [6]:
for split in ["train", "valid", "test"]:
    img_dir = os.path.join(DATASET_DIR, split, "images")
    lbl_dir = os.path.join(DATASET_DIR, split, "labels")
    n_img = len(os.listdir(img_dir)) if os.path.exists(img_dir) else 0
    n_lbl = len(os.listdir(lbl_dir)) if os.path.exists(lbl_dir) else 0
    print(f"{split:6s} -> images: {n_img:5d}  labels: {n_lbl:5d}")

train  -> images:   298  labels:   298
valid  -> images:    49  labels:    49
test   -> images:    25  labels:    25


## 7. Train YOLOv8 (50 epochs)

Using `yolov8n.pt` (nano) as the pretrained starting checkpoint for a good speed/accuracy tradeoff on Colab's free GPU. Swap to `yolov8s.pt` / `yolov8m.pt` if you want higher accuracy and have more GPU time.

Training outputs (weights, curves, confusion matrix) are saved under `runs/detect/football_players_yolov8/`.

In [8]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # pretrained nano model

results = model.train(
    data=colab_yaml_path,
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,          # early stopping patience
    project="runs/detect",
    name="football_players_yolov8",
    exist_ok=True,
    plots=True,
)

Ultralytics 8.4.92 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/football-players-detection/data_colab.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=football_players_yolov8, nbs=64, nms=False, opset

## 8. Validate the trained model on the validation set

In [10]:
import shutil

save_dir = os.path.join(DATASET_DIR, "trained_model")
os.makedirs(save_dir, exist_ok=True)

run_dir = "runs/detect/football_players_yolov8"
for fname in ["best.pt", "last.pt"]:
    src = os.path.join(run_dir, "weights", fname)
    if os.path.exists(src):
        shutil.copy(src, os.path.join(save_dir, fname))
        print(f"Saved {fname} -> {save_dir}")
    else:
        print(f"WARNING: {src} not found — training may not have completed.")

In [11]:
best_weights = "runs/detect/football_players_yolov8/weights/best.pt"

trained_model = YOLO(best_weights)
metrics = trained_model.val(data=colab_yaml_path, split="val")

print(f"mAP50:    {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")

FileNotFoundError: [Errno 2] No such file or directory: 'runs/detect/football_players_yolov8/weights/best.pt'

## 9. Run inference on the test set

In [13]:
test_results = trained_model.predict(
    source=os.path.join(DATASET_DIR, "test", "images"),
    conf=0.25,
    save=True,
    project="runs/detect",
    name="football_players_test_predictions",
    exist_ok=True,
)
print("Predictions saved to runs/detect/football_players_test_predictions/")

NameError: name 'trained_model' is not defined

## 10. Preview a few predictions

In [ ]:
import glob
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

pred_dir = "runs/detect/football_players_test_predictions"
pred_images = sorted(glob.glob(os.path.join(pred_dir, "*.jpg")))[:6]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, img_path in zip(axes.flatten(), pred_images):
    img = mpimg.imread(img_path)
    ax.imshow(img)
    ax.axis("off")
    ax.set_title(os.path.basename(img_path), fontsize=8)
plt.tight_layout()
plt.show()

## 11. View training curves

In [ ]:
results_png = "runs/detect/football_players_yolov8/results.png"
if os.path.exists(results_png):
    img = mpimg.imread(results_png)
    plt.figure(figsize=(14, 10))
    plt.imshow(img)
    plt.axis("off")
    plt.show()
else:
    print("results.png not found — check the training output folder.")

## 12. Save trained weights back to Google Drive

Copies `best.pt` (and `last.pt`) to your Drive so they persist after the Colab runtime disconnects.

In [ ]:
import shutil

save_dir = os.path.join(DATASET_DIR, "trained_model")
os.makedirs(save_dir, exist_ok=True)

for fname in ["best.pt", "last.pt"]:
    src = f"runs/detect/football_players_yolov8/weights/{fname}"
    if os.path.exists(src):
        shutil.copy(src, os.path.join(save_dir, fname))
        print(f"Saved {fname} -> {save_dir}")

# Also copy the full results folder (curves, confusion matrix, etc.)
results_dst = os.path.join(save_dir, "results")
if os.path.exists(results_dst):
    shutil.rmtree(results_dst)
shutil.copytree("runs/detect/football_players_yolov8", results_dst)
print(f"Saved full training results -> {results_dst}")

## 13. (Optional) Run inference on a single custom image

In [ ]:
# Example: run the trained model on any single image
# img_path = "/content/drive/MyDrive/football-players-detection/test/images/some_image.jpg"
# result = trained_model.predict(source=img_path, conf=0.25, save=True)
# result[0].show()